# Sistema de Gestión de Citas Médicas — Clínica San Gabriel

### Lenguaje de Programación II — Semestre 2026-2
**Evaluación Grupal 1 · Trabajo Parcial · Grupo 6**

---

| N° | Apellidos y nombres | Usuario de GitHub | Parte |
|---|---|---|---|
| 1 | Anaya Yarleque, Nahomi Alessandra | `@HanakoCrazy` | 1 — `Historial`, `Persona`, `Paciente` |
| 2 | Guerra Vilchez, Ximena Valery | `@ValeryGuerra7` | 2 — `Medico` y polimorfismo |
| 3 | Mendoza Chavez, Sergio | `@290803S` | 3 — `Cita` y reglas de negocio |
| 4 | Fernandez Leyva, Dayvi | `@dferl26` | 4 — `Clinica`, pandas y menú |

---

**Avance del archivo:** ■■■■ &nbsp; partes 1 a 4 de 4 · última parte agregada por **Dayvi Fernandez**.

**Con esta parte el proyecto queda terminado.**

---

### El escenario

La Clínica San Gabriel llevaba el control de sus citas en cuadernos, lo que producía tres problemas: se programaban **dos pacientes con el mismo médico a la misma hora**, el **historial clínico se perdía** y **no existían reportes**.

El sistema resuelve los tres modelando cada elemento del proceso como un objeto: `Paciente`, `Medico`, `Cita`, `Historial` y `Clinica`, todas construidas sobre la clase base `Persona`.

> Ejecute el cuaderno completo con **Entorno de ejecución → Ejecutar todas**.

---

# Parte 1 — Anaya Yarleque, Nahomi Alessandra

Las clases base del sistema. Aquí se definen los datos que comparten todas las personas y la estructura que guarda los antecedentes médicos.

## 1.1 Clase `Historial`

Guarda las atenciones de un paciente. Cada atención es un **diccionario**, para poder convertirlas después en una tabla de pandas sin trabajo adicional.

Se define primero porque `Paciente` la necesita: al crear un paciente se le crea también su historial.

In [1]:
# ------------------------------------------------------------
#  Clase Historial
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------
# Guarda las atenciones de UN paciente. Cada atencion se guarda
# como un diccionario, para poder convertirlas despues en una
# tabla de pandas sin trabajo adicional.


class Historial:
    # Atributo de CLASE: el mismo para todos los historiales
    centro = "Clinica San Gabriel"

    def __init__(self, codigo_paciente):
        # Atributos de INSTANCIA: distintos en cada objeto
        self.codigo_paciente = codigo_paciente
        self.atenciones = []

    def agregar_atencion(self, fecha, especialidad, medico, diagnostico, tratamiento):
        atencion = {
            "fecha": fecha,
            "especialidad": especialidad,
            "medico": medico,
            "diagnostico": diagnostico,
            "tratamiento": tratamiento
        }
        self.atenciones.append(atencion)

    def total_atenciones(self):
        return len(self.atenciones)

    def ultimo_diagnostico(self):
        if len(self.atenciones) == 0:
            return "Sin atenciones registradas"
        ultima = self.atenciones[-1]
        return ultima["diagnostico"]

    def mostrar(self):
        print("HISTORIAL CLINICO DEL PACIENTE " + self.codigo_paciente)
        if len(self.atenciones) == 0:
            print("   (Sin atenciones registradas)")
        else:
            numero = 1
            for a in self.atenciones:
                print("   " + str(numero) + ". " + a["fecha"] + " | " + a["especialidad"] +
                      " | Dx: " + a["diagnostico"] + " | Tto: " + a["tratamiento"])
                numero = numero + 1
        print("   Total de atenciones: " + str(self.total_atenciones()))


print("Clase Historial creada.")

Clase Historial creada.


## 1.2 Clase base `Persona`

Reúne los cuatro datos que comparten pacientes y médicos. Se usan **getters** para leer y **setters** para modificar: el setter revisa el dato antes de guardarlo, por eso `set_telefono("123")` no modifica nada y avisa del error.

El método `mostrar_informacion()` se escribe aquí de forma general y las clases hijas lo van a **sobrescribir**.

In [2]:
# ------------------------------------------------------------
#  Clase base Persona
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------
# Reune los datos que comparten pacientes y medicos. De ella van
# a HEREDAR las clases Paciente y Medico.


class Persona:
    def __init__(self, nombre, apellido, dni, telefono):
        self.nombre = nombre
        self.apellido = apellido
        self.dni = dni
        self.telefono = telefono

    # ---------- Getters: leer los datos ----------
    def get_nombre_completo(self):
        return self.nombre + " " + self.apellido

    def get_dni(self):
        return self.dni

    # ---------- Setter: modificar con validacion ----------
    def set_telefono(self, nuevo_telefono):
        """Cambia el telefono solo si tiene 9 digitos numericos."""
        if len(nuevo_telefono) == 9 and nuevo_telefono.isdigit():
            self.telefono = nuevo_telefono
            print("Telefono actualizado a " + nuevo_telefono)
        else:
            print("ERROR: el telefono debe tener 9 digitos numericos.")

    # ---------- Metodo que las clases hijas van a sobrescribir ----------
    def mostrar_informacion(self):
        return "[PERSONA]  " + self.get_nombre_completo() + " | DNI: " + self.dni

    def __str__(self):
        return self.get_nombre_completo()


print("Clase Persona creada.")

Clase Persona creada.


## 1.3 Clase `Paciente` (hereda de `Persona`)

Tres mecanismos trabajando en el mismo constructor:

1. **`super().__init__(...)`** llama al constructor del padre. Sin esa línea habría que repetir las cuatro asignaciones de los datos personales en cada clase hija.
2. **`contador`** es un atributo de clase que se incrementa cada vez que nace un paciente, y genera los códigos `P001`, `P002`, `P003`.
3. **`self.historial = Historial(...)`** es **composición**: un objeto que vive dentro de otro.

In [3]:
# ------------------------------------------------------------
#  Clase Paciente (hereda de Persona)
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------


class Paciente(Persona):
    # Atributo de CLASE: genera los codigos P001, P002, P003...
    contador = 0

    def __init__(self, nombre, apellido, dni, telefono, edad, tiene_seguro=False):
        # super() reutiliza el constructor del padre
        super().__init__(nombre, apellido, dni, telefono)

        self.edad = edad
        self.tiene_seguro = tiene_seguro

        Paciente.contador = Paciente.contador + 1
        self.codigo = "P" + str(Paciente.contador).zfill(3)

        # COMPOSICION: cada paciente nace con su propio historial
        self.historial = Historial(self.codigo)

    def get_codigo(self):
        return self.codigo

    def set_edad(self, nueva_edad):
        if nueva_edad > 0 and nueva_edad < 120:
            self.edad = nueva_edad
        else:
            print("ERROR: la edad debe estar entre 1 y 119 anios.")

    def es_adulto_mayor(self):
        """Los pacientes de 60 anios a mas tienen atencion preferente."""
        return self.edad >= 60

    def a_diccionario(self):
        """Convierte el objeto en un diccionario, para usarlo con pandas."""
        return {
            "codigo": self.codigo,
            "nombre": self.get_nombre_completo(),
            "edad": self.edad,
            "tiene_seguro": self.tiene_seguro,
            "atenciones": self.historial.total_atenciones()
        }

    # ---------- Sobrescritura (override) ----------
    def mostrar_informacion(self):
        if self.tiene_seguro:
            seguro = "SI"
        else:
            seguro = "NO"

        texto = ("[PACIENTE] " + self.codigo + " | " + self.get_nombre_completo() +
                 " | Edad: " + str(self.edad) + " | Seguro: " + seguro)

        if self.es_adulto_mayor():
            texto = texto + " | Atencion preferente"
        return texto

    def __str__(self):
        return self.codigo + " - " + self.get_nombre_completo()


print("Clase Paciente creada.")

Clase Paciente creada.


## 1.4 Prueba de la parte 1

In [4]:
# ------------------------------------------------------------
#  Prueba de la parte 1
# ------------------------------------------------------------

print("=" * 58)
print("PRUEBA DE LA PARTE 1")
print("=" * 58)

# ---------- Creacion de objetos ----------
print("\n1. PACIENTES REGISTRADOS")
print("-" * 58)
p1 = Paciente("Nahomi", "Anaya Yarleque", "72345678", "900111222", 23, True)
p2 = Paciente("Ximena", "Guerra Vilchez", "73456789", "900333444", 24)
p3 = Paciente("Luis", "Ramos Diaz", "74567890", "900555666", 67, True)

for paciente in [p1, p2, p3]:
    print(paciente.mostrar_informacion())

# ---------- Atributo de clase ----------
print("\n2. ATRIBUTO DE CLASE")
print("-" * 58)
print("Pacientes creados (Paciente.contador):", Paciente.contador)
print("Codigo del primero (atributo de instancia):", p1.get_codigo())

# ---------- Setters con validacion ----------
print("\n3. SETTERS CON VALIDACION")
print("-" * 58)
p1.set_telefono("123")
p1.set_telefono("955123456")
p2.set_edad(200)

# ---------- Herencia ----------
print("\n4. HERENCIA")
print("-" * 58)
print("isinstance(p1, Paciente):", isinstance(p1, Paciente))
print("isinstance(p1, Persona) :", isinstance(p1, Persona))
print("get_dni() esta escrito en Persona y el paciente lo usa:", p1.get_dni())

# ---------- Composicion ----------
print("\n5. COMPOSICION: CADA PACIENTE TIENE SU HISTORIAL")
print("-" * 58)
p1.historial.mostrar()
print("\nSe agrega una atencion de prueba:")
p1.historial.agregar_atencion("20/10/2026", "Medicina General",
                              "Carlos Rojas", "Faringitis aguda", "Amoxicilina 500mg")
p1.historial.mostrar()

PRUEBA DE LA PARTE 1

1. PACIENTES REGISTRADOS
----------------------------------------------------------
[PACIENTE] P001 | Nahomi Anaya Yarleque | Edad: 23 | Seguro: SI
[PACIENTE] P002 | Ximena Guerra Vilchez | Edad: 24 | Seguro: NO
[PACIENTE] P003 | Luis Ramos Diaz | Edad: 67 | Seguro: SI | Atencion preferente

2. ATRIBUTO DE CLASE
----------------------------------------------------------
Pacientes creados (Paciente.contador): 3
Codigo del primero (atributo de instancia): P001

3. SETTERS CON VALIDACION
----------------------------------------------------------
ERROR: el telefono debe tener 9 digitos numericos.
Telefono actualizado a 955123456
ERROR: la edad debe estar entre 1 y 119 anios.

4. HERENCIA
----------------------------------------------------------
isinstance(p1, Paciente): True
isinstance(p1, Persona) : True
get_dni() esta escrito en Persona y el paciente lo usa: 72345678

5. COMPOSICION: CADA PACIENTE TIENE SU HISTORIAL
-------------------------------------------------

---

# Parte 2 — Guerra Vilchez, Ximena Valery

## 2.1 Clase `Medico` (hereda de `Persona`)

Agrega especialidad, tarifa y **agenda**. La agenda es una lista de textos como `"26/10/2026 09:00"`, y preguntar si el médico está libre es preguntar si ese texto está en la lista:

```python
def esta_disponible(self, fecha, hora):
    horario = fecha + " " + hora
    return horario not in self.agenda
```

Son dos líneas, pero son las que cumplen el **valor añadido** del enunciado: evitar que se agenden dos citas al mismo médico en la misma hora.

In [5]:
# ------------------------------------------------------------
#  Clase Medico (hereda de Persona)
#  Integrante: Guerra Vilchez, Ximena Valery
# ------------------------------------------------------------
# Ademas de los datos personales guarda su AGENDA, que es la
# lista de horarios ya ocupados. Es la base del valor anadido.


class Medico(Persona):
    # Atributo de CLASE: genera los codigos M001, M002, M003...
    contador = 0

    def __init__(self, nombre, apellido, dni, telefono, especialidad, tarifa):
        super().__init__(nombre, apellido, dni, telefono)

        self.especialidad = especialidad
        self.tarifa = tarifa
        self.agenda = []          # lista de horarios reservados

        Medico.contador = Medico.contador + 1
        self.codigo = "M" + str(Medico.contador).zfill(3)

    # ---------- Getters ----------
    def get_codigo(self):
        return self.codigo

    def get_especialidad(self):
        return self.especialidad

    def get_tarifa(self):
        return self.tarifa

    # ---------- Setter ----------
    def set_tarifa(self, nueva_tarifa):
        if nueva_tarifa > 0:
            self.tarifa = nueva_tarifa
            print("Tarifa actualizada a S/ " + str(nueva_tarifa))
        else:
            print("ERROR: la tarifa debe ser mayor que cero.")

    # ---------- La agenda: el valor anadido del proyecto ----------
    def esta_disponible(self, fecha, hora):
        """Responde si el medico tiene libre ese dia y esa hora."""
        horario = fecha + " " + hora
        return horario not in self.agenda

    def ocupar_horario(self, fecha, hora):
        horario = fecha + " " + hora
        if self.esta_disponible(fecha, hora):
            self.agenda.append(horario)
            return True
        else:
            print("ERROR: el Dr(a). " + self.get_nombre_completo() +
                  " ya tiene una cita el " + horario + ".")
            return False

    def liberar_horario(self, fecha, hora):
        horario = fecha + " " + hora
        if horario in self.agenda:
            self.agenda.remove(horario)
            return True
        return False

    def total_citas(self):
        return len(self.agenda)

    # ---------- Sobrescritura (override) ----------
    def mostrar_informacion(self):
        return ("[MEDICO]   " + self.codigo + " | Dr(a). " + self.get_nombre_completo() +
                " | " + self.especialidad + " | S/ " + str(self.tarifa) +
                " | Citas: " + str(self.total_citas()))

    def __str__(self):
        return self.codigo + " - Dr(a). " + self.get_nombre_completo() + " (" + self.especialidad + ")"


print("Clase Medico creada.")

Clase Medico creada.


## 2.2 Prueba de la parte 2

Incluye la demostración de **polimorfismo**: una sola lista guarda pacientes y médicos, y al llamar al mismo método cada objeto responde según su clase, sin un solo `if` de tipo.

In [6]:
# ------------------------------------------------------------
#  Prueba de la parte 2
# ------------------------------------------------------------

print("=" * 58)
print("PRUEBA DE LA PARTE 2")
print("=" * 58)

# ---------- Creacion de medicos ----------
print("\n1. MEDICOS REGISTRADOS")
print("-" * 58)
m1 = Medico("Carlos", "Rojas Vega", "40112233", "987654321", "Medicina General", 60)
m2 = Medico("Lucia", "Paredes Soto", "41223344", "987111222", "Pediatria", 90)
m3 = Medico("Javier", "Quispe Luna", "42334455", "987333444", "Cardiologia", 150)

for medico in [m1, m2, m3]:
    print(medico.mostrar_informacion())

# ---------- Setter con validacion ----------
print("\n2. SETTER CON VALIDACION")
print("-" * 58)
m1.set_tarifa(-50)
m1.set_tarifa(70)

# ---------- El valor anadido ----------
print("\n3. CONTROL DE LA AGENDA (valor anadido)")
print("-" * 58)
print("Libre el 26/10/2026 a las 09:00?:", m1.esta_disponible("26/10/2026", "09:00"))

print("\nSe reserva ese horario:")
m1.ocupar_horario("26/10/2026", "09:00")
print("Libre ahora?:", m1.esta_disponible("26/10/2026", "09:00"))

print("\nSe intenta reservar el MISMO horario otra vez:")
m1.ocupar_horario("26/10/2026", "09:00")

print("\nSe libera el horario:")
m1.liberar_horario("26/10/2026", "09:00")
print("Libre de nuevo?:", m1.esta_disponible("26/10/2026", "09:00"))

# ---------- Polimorfismo ----------
print("\n4. POLIMORFISMO")
print("-" * 58)
print("Una sola lista con pacientes y medicos. Todos responden al")
print("mismo metodo mostrar_informacion(), cada uno a su manera:\n")

personas = [p1, m1, p3, m3]
for elemento in personas:
    print(type(elemento).__name__.ljust(9), "->", elemento.mostrar_informacion())

print("\nNo hizo falta ningun 'if' para saber de que tipo era cada objeto.")

PRUEBA DE LA PARTE 2

1. MEDICOS REGISTRADOS
----------------------------------------------------------
[MEDICO]   M001 | Dr(a). Carlos Rojas Vega | Medicina General | S/ 60 | Citas: 0
[MEDICO]   M002 | Dr(a). Lucia Paredes Soto | Pediatria | S/ 90 | Citas: 0
[MEDICO]   M003 | Dr(a). Javier Quispe Luna | Cardiologia | S/ 150 | Citas: 0

2. SETTER CON VALIDACION
----------------------------------------------------------
ERROR: la tarifa debe ser mayor que cero.
Tarifa actualizada a S/ 70

3. CONTROL DE LA AGENDA (valor anadido)
----------------------------------------------------------
Libre el 26/10/2026 a las 09:00?: True

Se reserva ese horario:
Libre ahora?: False

Se intenta reservar el MISMO horario otra vez:
ERROR: el Dr(a). Carlos Rojas Vega ya tiene una cita el 26/10/2026 09:00.

Se libera el horario:
Libre de nuevo?: True

4. POLIMORFISMO
----------------------------------------------------------
Una sola lista con pacientes y medicos. Todos responden al
mismo metodo mostrar_i

---

# Parte 3 — Mendoza Chavez, Sergio

## 3.1 Clase `Cita`

Relaciona un `Paciente` con un `Medico`. Guarda los **objetos completos**, no sus códigos, para poder preguntarles directamente: `self.medico.get_tarifa()`.

Sus tres atributos de clase (`contador`, `descuento_seguro`, `estados`) valen lo mismo para todas las citas: si la clínica cambia el descuento al 25 %, se modifica una línea.

### Ciclo de vida de una cita

```
                  atender()
  PENDIENTE  ──────────────────▶  ATENDIDA    (escribe en el historial)
      │ cancelar()
      ▼
  CANCELADA    (libera el horario del médico)
```

Las dos transiciones son definitivas: los métodos revisan el estado actual antes de cambiar nada, por eso una cita no se puede atender dos veces ni cancelar después de atendida.

El método `atender()` hace tres cosas en un solo llamado: cambia el estado, guarda el diagnóstico y **escribe en el historial del paciente**. Gracias a lo último es imposible atender una cita y olvidar el registro.

In [7]:
# ------------------------------------------------------------
#  Clase Cita
#  Integrante: Mendoza Chavez, Sergio
# ------------------------------------------------------------
# Relaciona un objeto Paciente con un objeto Medico en una fecha
# y una hora. Es la clase que une todo el sistema.


class Cita:
    # ---------- ATRIBUTOS DE CLASE ----------
    contador = 0
    descuento_seguro = 0.20
    estados = ["PENDIENTE", "ATENDIDA", "CANCELADA"]

    def __init__(self, paciente, medico, fecha, hora, motivo):
        Cita.contador = Cita.contador + 1
        self.codigo = "C" + str(Cita.contador).zfill(3)

        # Se guardan los OBJETOS completos, no solo sus codigos
        self.paciente = paciente
        self.medico = medico

        self.fecha = fecha
        self.hora = hora
        self.motivo = motivo

        self.estado = "PENDIENTE"      # toda cita nueva nace pendiente
        self.diagnostico = ""
        self.tratamiento = ""

    def get_codigo(self):
        return self.codigo

    def get_estado(self):
        return self.estado

    def costo(self):
        """Le pregunta la tarifa al medico y el seguro al paciente."""
        precio = self.medico.get_tarifa()
        if self.paciente.tiene_seguro:
            precio = precio - (precio * Cita.descuento_seguro)
        return round(precio, 2)

    def atender(self, diagnostico, tratamiento):
        """Cambia el estado y escribe en el historial del paciente."""
        if self.estado != "PENDIENTE":
            print("ERROR: la cita " + self.codigo + " ya fue " + self.estado.lower() + ".")
            return False

        if len(diagnostico) < 4:
            print("ERROR: el diagnostico es demasiado corto.")
            return False

        self.diagnostico = diagnostico
        self.tratamiento = tratamiento
        self.estado = "ATENDIDA"

        # Escribe en el historial del paciente
        self.paciente.historial.agregar_atencion(
            self.fecha,
            self.medico.get_especialidad(),
            self.medico.get_nombre_completo(),
            diagnostico,
            tratamiento
        )

        print("La cita " + self.codigo + " fue atendida y se registro en el historial.")
        return True

    def cancelar(self):
        """Cancela la cita y libera el horario del medico."""
        if self.estado == "ATENDIDA":
            print("ERROR: no se puede cancelar una cita que ya fue atendida.")
            return False
        if self.estado == "CANCELADA":
            print("ERROR: la cita " + self.codigo + " ya estaba cancelada.")
            return False

        self.estado = "CANCELADA"
        self.medico.liberar_horario(self.fecha, self.hora)
        print("La cita " + self.codigo + " fue cancelada y el horario quedo libre.")
        return True

    def comprobante(self):
        if self.estado != "ATENDIDA":
            return "La cita " + self.codigo + " no genera comprobante (estado: " + self.estado + ")."

        linea = "+" + "-" * 48 + "+"
        t = linea + "\n"
        t = t + "|          COMPROBANTE DE ATENCION               |\n"
        t = t + linea + "\n"
        t = t + " Cita       : " + self.codigo + "\n"
        t = t + " Fecha      : " + self.fecha + "   " + self.hora + "\n"
        t = t + " Paciente   : " + self.paciente.get_nombre_completo() + "\n"
        t = t + " Medico     : Dr(a). " + self.medico.get_nombre_completo() + "\n"
        t = t + " Servicio   : " + self.medico.get_especialidad() + "\n"
        t = t + " Diagnostico: " + self.diagnostico + "\n"
        t = t + " " + "-" * 47 + "\n"
        t = t + " Tarifa     : S/ " + str(self.medico.get_tarifa()) + "\n"
        if self.paciente.tiene_seguro:
            descuento = self.medico.get_tarifa() * Cita.descuento_seguro
            t = t + " Seguro -20%: S/ " + str(round(descuento, 2)) + "\n"
        t = t + " TOTAL      : S/ " + str(self.costo()) + "\n"
        t = t + linea
        return t

    def a_diccionario(self):
        """Convierte la cita en un diccionario, para usarlo con pandas."""
        return {
            "codigo": self.codigo,
            "fecha": self.fecha,
            "paciente": self.paciente.get_nombre_completo(),
            "medico": self.medico.get_nombre_completo(),
            "especialidad": self.medico.get_especialidad(),
            "estado": self.estado,
            "costo": self.costo()
        }

    def __str__(self):
        return (self.codigo + " | " + self.fecha + " " + self.hora +
                " | " + self.paciente.get_nombre_completo() +
                " | Dr(a). " + self.medico.get_nombre_completo() +
                " | " + self.estado)


print("Clase Cita creada.")

Clase Cita creada.


## 3.2 Prueba de la parte 3

In [8]:
# ------------------------------------------------------------
#  Prueba de la parte 3
# ------------------------------------------------------------

print("=" * 58)
print("PRUEBA DE LA PARTE 3")
print("=" * 58)

# ---------- Atributos de clase ----------
print("\n1. ATRIBUTOS DE CLASE")
print("-" * 58)
print("Cita.estados          :", Cita.estados)
print("Cita.descuento_seguro :", Cita.descuento_seguro)

# ---------- Creacion de citas ----------
print("\n2. CITAS CREADAS")
print("-" * 58)
c1 = Cita(p1, m1, "26/10/2026", "09:00", "Control general")
m1.ocupar_horario("26/10/2026", "09:00")

c2 = Cita(p2, m2, "26/10/2026", "10:00", "Dolor de garganta")
m2.ocupar_horario("26/10/2026", "10:00")

for cita in [c1, c2]:
    print(" ", cita)
print("\nEstado inicial de toda cita nueva:", c1.get_estado())

# ---------- El costo ----------
print("\n3. COSTO CON Y SIN SEGURO")
print("-" * 58)
print("Tarifa del Dr(a).", m1.get_nombre_completo(), ": S/", m1.get_tarifa())
print(p1.get_nombre_completo(), "tiene seguro:", p1.tiene_seguro)
print("Costo de", c1.get_codigo(), ": S/", c1.costo(), " <- con 20% de descuento")
print()
print(p2.get_nombre_completo(), "tiene seguro:", p2.tiene_seguro)
print("Costo de", c2.get_codigo(), ": S/", c2.costo())

# ---------- Atender ----------
print("\n4. ATENDER UNA CITA")
print("-" * 58)
print("Atenciones en el historial ANTES :", p1.historial.total_atenciones())
c1.atender("Faringitis aguda", "Amoxicilina 500mg por 7 dias")
print("Atenciones en el historial DESPUES:", p1.historial.total_atenciones())
print("Estado de la cita                 :", c1.get_estado())

# ---------- Comprobante ----------
print("\n5. COMPROBANTE")
print("-" * 58)
print(c1.comprobante())

# ---------- Cancelar ----------
print("\n6. CANCELAR UNA CITA")
print("-" * 58)
print("Agenda del Dr(a).", m2.get_nombre_completo(), "antes:", m2.total_citas())
c2.cancelar()
print("Agenda despues:", m2.total_citas())

# ---------- Operaciones invalidas ----------
print("\n7. OPERACIONES INVALIDAS")
print("-" * 58)
print("a) Atender una cita ya atendida:")
c1.atender("Otro diagnostico", "Otro tratamiento")
print("\nb) Cancelar una cita ya atendida:")
c1.cancelar()
print("\nc) Cancelar dos veces la misma cita:")
c2.cancelar()
print("\nEl programa rechazo las tres y siguio funcionando.")

PRUEBA DE LA PARTE 3

1. ATRIBUTOS DE CLASE
----------------------------------------------------------
Cita.estados          : ['PENDIENTE', 'ATENDIDA', 'CANCELADA']
Cita.descuento_seguro : 0.2

2. CITAS CREADAS
----------------------------------------------------------
  C001 | 26/10/2026 09:00 | Nahomi Anaya Yarleque | Dr(a). Carlos Rojas Vega | PENDIENTE
  C002 | 26/10/2026 10:00 | Ximena Guerra Vilchez | Dr(a). Lucia Paredes Soto | PENDIENTE

Estado inicial de toda cita nueva: PENDIENTE

3. COSTO CON Y SIN SEGURO
----------------------------------------------------------
Tarifa del Dr(a). Carlos Rojas Vega : S/ 70
Nahomi Anaya Yarleque tiene seguro: True
Costo de C001 : S/ 56.0  <- con 20% de descuento

Ximena Guerra Vilchez tiene seguro: False
Costo de C002 : S/ 90

4. ATENDER UNA CITA
----------------------------------------------------------
Atenciones en el historial ANTES : 1
La cita C001 fue atendida y se registro en el historial.
Atenciones en el historial DESPUES: 2
Estado 

---

# Parte 4 — Fernandez Leyva, Dayvi

## 4.1 Clase `Clinica` (controladora)

Guarda tres listas de objetos (`pacientes`, `medicos`, `citas`) y concentra las reglas del negocio.

Su método `agendar_cita()` verifica **cinco reglas** antes de crear la cita:

| # | Regla | Cómo se verifica |
|---|---|---|
| 1 | El paciente debe existir | `buscar_paciente(dni)` |
| 2 | El médico debe existir | `buscar_medico(codigo)` |
| 3 | La hora debe estar entre 08:00 y 18:00 | `int(hora[0:2])` |
| 4 | **El médico debe tener libre ese horario** | `medico.esta_disponible(...)` |
| 5 | El paciente no puede tener dos citas a la misma hora | recorre `self.citas` |

La regla 4 es el valor añadido del enunciado. Note que `Clinica` no revisa la agenda por su cuenta: le **pregunta** al médico.

In [9]:
# ------------------------------------------------------------
#  Clase Clinica (clase controladora)
#  Integrante: Fernandez Leyva, Dayvi
# ------------------------------------------------------------
# Guarda las tres listas de objetos y concentra las reglas.


class Clinica:
    def __init__(self, nombre, ruc):
        self.nombre = nombre
        self.ruc = ruc

        # ---------- ESTRUCTURAS DE DATOS ----------
        self.pacientes = []
        self.medicos = []
        self.citas = []

    # ==========================================================
    #  REGISTRO
    # ==========================================================
    def registrar_paciente(self, nombre, apellido, dni, telefono, edad, tiene_seguro=False):
        if len(dni) != 8 or not dni.isdigit():
            print("ERROR: el DNI debe tener 8 digitos numericos.")
            return None

        if self.buscar_paciente(dni) is not None:
            print("ERROR: ya existe un paciente con el DNI " + dni + ".")
            return None

        paciente = Paciente(nombre, apellido, dni, telefono, edad, tiene_seguro)
        self.pacientes.append(paciente)
        print("Paciente registrado: " + paciente.get_codigo() + " - " + paciente.get_nombre_completo())
        return paciente

    def registrar_medico(self, nombre, apellido, dni, telefono, especialidad, tarifa):
        if len(dni) != 8 or not dni.isdigit():
            print("ERROR: el DNI debe tener 8 digitos numericos.")
            return None

        if tarifa <= 0:
            print("ERROR: la tarifa debe ser mayor que cero.")
            return None

        medico = Medico(nombre, apellido, dni, telefono, especialidad, tarifa)
        self.medicos.append(medico)
        print("Medico registrado: " + medico.get_codigo() + " - Dr(a). " + medico.get_nombre_completo())
        return medico

    # ==========================================================
    #  BUSQUEDAS
    # ==========================================================
    def buscar_paciente(self, dni):
        for paciente in self.pacientes:
            if paciente.get_dni() == dni:
                return paciente
        return None

    def buscar_medico(self, codigo):
        for medico in self.medicos:
            if medico.get_codigo() == codigo.upper():
                return medico
        return None

    def buscar_cita(self, codigo):
        for cita in self.citas:
            if cita.get_codigo() == codigo.upper():
                return cita
        return None

    # ==========================================================
    #  GESTION DE CITAS
    # ==========================================================
    def agendar_cita(self, dni_paciente, codigo_medico, fecha, hora, motivo):
        """Programa una cita revisando las cinco reglas del negocio."""

        # REGLA 1: el paciente debe existir
        paciente = self.buscar_paciente(dni_paciente)
        if paciente is None:
            print("ERROR: no existe un paciente con DNI " + dni_paciente + ".")
            return None

        # REGLA 2: el medico debe existir
        medico = self.buscar_medico(codigo_medico)
        if medico is None:
            print("ERROR: no existe un medico con codigo " + codigo_medico + ".")
            return None

        # REGLA 3: horario de atencion de 08:00 a 18:00
        hora_numero = int(hora[0:2])
        if hora_numero < 8 or hora_numero >= 18:
            print("ERROR: la clinica atiende de 08:00 a 18:00.")
            return None

        # REGLA 4: el medico debe tener libre ese horario  <- VALOR ANADIDO
        if not medico.esta_disponible(fecha, hora):
            print("ERROR: el Dr(a). " + medico.get_nombre_completo() +
                  " ya tiene una cita el " + fecha + " a las " + hora + ".")
            return None

        # REGLA 5: el paciente no puede tener dos citas a la misma hora
        for cita in self.citas:
            if (cita.paciente.get_dni() == dni_paciente and
                    cita.fecha == fecha and cita.hora == hora and
                    cita.get_estado() == "PENDIENTE"):
                print("ERROR: " + paciente.get_nombre_completo() +
                      " ya tiene la cita " + cita.get_codigo() + " a esa misma hora.")
                return None

        # Si paso las cinco reglas, recien se crea la cita
        cita = Cita(paciente, medico, fecha, hora, motivo)
        medico.ocupar_horario(fecha, hora)
        self.citas.append(cita)
        print("Cita " + cita.get_codigo() + " programada para el " + fecha +
              " a las " + hora + ". Costo: S/ " + str(cita.costo()))
        return cita

    def atender_cita(self, codigo_cita, diagnostico, tratamiento):
        cita = self.buscar_cita(codigo_cita)
        if cita is None:
            print("ERROR: no existe la cita " + codigo_cita + ".")
            return False
        return cita.atender(diagnostico, tratamiento)

    def cancelar_cita(self, codigo_cita):
        cita = self.buscar_cita(codigo_cita)
        if cita is None:
            print("ERROR: no existe la cita " + codigo_cita + ".")
            return False
        return cita.cancelar()

    def listar_citas(self, estado=None):
        if estado is None:
            return self.citas
        resultado = []
        for cita in self.citas:
            if cita.get_estado() == estado.upper():
                resultado.append(cita)
        return resultado

    # ==========================================================
    #  REPORTES
    # ==========================================================
    def ingresos_totales(self):
        total = 0
        for cita in self.citas:
            if cita.get_estado() == "ATENDIDA":
                total = total + cita.costo()
        return round(total, 2)

    def medico_mas_solicitado(self):
        if len(self.medicos) == 0:
            return None
        mejor = self.medicos[0]
        for medico in self.medicos:
            if medico.total_citas() > mejor.total_citas():
                mejor = medico
        if mejor.total_citas() == 0:
            return None
        return mejor

    def contar_por_estado(self):
        resumen = {}
        for estado in Cita.estados:
            resumen[estado] = len(self.listar_citas(estado))
        return resumen

    def reporte_general(self):
        print("=" * 58)
        print("REPORTE GENERAL - " + self.nombre.upper())
        print("=" * 58)
        print("RUC                   : " + self.ruc)
        print("Pacientes registrados : " + str(len(self.pacientes)))
        print("Medicos registrados   : " + str(len(self.medicos)))
        print("Citas registradas     : " + str(len(self.citas)))

        print("\nCITAS POR ESTADO")
        print("-" * 58)
        resumen = self.contar_por_estado()
        for estado in resumen:
            print("  " + estado.ljust(10) + str(resumen[estado]).rjust(3) + "  " + "#" * resumen[estado])

        print("\nINGRESO TOTAL         : S/ " + str(self.ingresos_totales()))

        destacado = self.medico_mas_solicitado()
        if destacado is None:
            print("MEDICO MAS SOLICITADO : (aun no hay citas)")
        else:
            print("MEDICO MAS SOLICITADO : Dr(a). " + destacado.get_nombre_completo() +
                  " (" + destacado.get_especialidad() + ") con " +
                  str(destacado.total_citas()) + " cita(s)")

    def tabla_citas(self):
        """Lista de diccionarios, lista para pasar a pandas."""
        return [cita.a_diccionario() for cita in self.citas]

    def __str__(self):
        return self.nombre + " (RUC " + self.ruc + ")"


print("Clase Clinica creada.")

Clase Clinica creada.


## 4.2 Ejecución de prueba del sistema completo

Recorre todo el ciclo: registrar, agendar, rechazar operaciones inválidas, atender, cancelar y reportar.

In [10]:
# ------------------------------------------------------------
#  Ejecucion de prueba del sistema completo
# ------------------------------------------------------------

# Los contadores son atributos de CLASE y siguen sumando desde
# las pruebas anteriores. Se reinician para que la demostracion
# final empiece en P001, M001 y C001.
Paciente.contador = 0
Medico.contador = 0
Cita.contador = 0

print("=" * 58)
print("EJECUCION DE PRUEBA DEL SISTEMA COMPLETO")
print("=" * 58)

# ---------- 1. Registro ----------
print("\n1. REGISTRO DE DATOS")
print("-" * 58)
clinica = Clinica("Clinica San Gabriel", "20512345678")
print(clinica, "\n")

clinica.registrar_medico("Carlos", "Rojas Vega", "40112233", "987654321", "Medicina General", 60)
clinica.registrar_medico("Lucia", "Paredes Soto", "41223344", "987111222", "Pediatria", 90)
clinica.registrar_medico("Javier", "Quispe Luna", "42334455", "987333444", "Cardiologia", 150)
print()
clinica.registrar_paciente("Nahomi", "Anaya Yarleque", "72345678", "900111222", 23, True)
clinica.registrar_paciente("Ximena", "Guerra Vilchez", "73456789", "900333444", 24)
clinica.registrar_paciente("Sergio", "Mendoza Chavez", "74567890", "900555666", 67, True)
clinica.registrar_paciente("Dayvi", "Fernandez Leyva", "75678901", "900777888", 22)

print("\n2. REGISTROS INVALIDOS")
print("-" * 58)
clinica.registrar_paciente("Ana", "Torres", "1234", "900000000", 30)
clinica.registrar_paciente("Otro", "Paciente", "72345678", "900000000", 30)
clinica.registrar_medico("Pedro", "Salas", "43445566", "987000111", "Dermatologia", -50)

# ---------- 3. Programacion de citas ----------
print("\n3. PROGRAMACION DE CITAS")
print("-" * 58)
clinica.agendar_cita("72345678", "M001", "26/10/2026", "09:00", "Control general")
clinica.agendar_cita("73456789", "M002", "26/10/2026", "10:00", "Dolor de garganta")
clinica.agendar_cita("74567890", "M003", "27/10/2026", "11:30", "Chequeo cardiaco")
clinica.agendar_cita("75678901", "M001", "27/10/2026", "09:00", "Certificado medico")
clinica.agendar_cita("72345678", "M003", "28/10/2026", "16:00", "Evaluacion de presion")

# ---------- 4. El valor anadido ----------
print("\n4. VALOR ANADIDO: NO SE PERMITEN CITAS CRUZADAS")
print("-" * 58)
print("a) Otro paciente quiere al MISMO medico el MISMO dia y hora:")
clinica.agendar_cita("73456789", "M001", "26/10/2026", "09:00", "Consulta general")

print("\nb) La MISMA paciente quiere dos citas a la misma hora:")
clinica.agendar_cita("72345678", "M002", "26/10/2026", "09:00", "Consulta general")

print("\nc) Una cita fuera del horario de atencion:")
clinica.agendar_cita("72345678", "M002", "26/10/2026", "21:00", "Consulta general")

print("\nd) Un paciente que no esta registrado:")
clinica.agendar_cita("99999999", "M001", "26/10/2026", "15:00", "Consulta general")

print("\ne) Un medico que no existe:")
clinica.agendar_cita("72345678", "M099", "26/10/2026", "15:00", "Consulta general")

# ---------- 5. Atencion ----------
print("\n5. ATENCION DE LAS CITAS")
print("-" * 58)
clinica.atender_cita("C001", "Faringitis aguda", "Amoxicilina 500mg por 7 dias")
clinica.atender_cita("C002", "Amigdalitis viral", "Reposo e hidratacion")
clinica.atender_cita("C003", "Hipertension leve", "Dieta baja en sodio")
clinica.atender_cita("C005", "Presion arterial normal", "Continuar tratamiento")

print("\nComprobante de la cita C001:")
print(clinica.buscar_cita("C001").comprobante())

# ---------- 6. Cancelacion ----------
print("\n6. CANCELACION Y REUTILIZACION DEL HORARIO")
print("-" * 58)
medico_m001 = clinica.buscar_medico("M001")
print("Agenda del Dr(a).", medico_m001.get_nombre_completo(), "antes:", medico_m001.total_citas())
clinica.cancelar_cita("C004")
print("Agenda despues de cancelar:", medico_m001.total_citas())
print("\nEse horario libre lo puede tomar otro paciente:")
clinica.agendar_cita("73456789", "M001", "27/10/2026", "09:00", "Control de presion")

# ---------- 7. Historiales ----------
print("\n7. EL HISTORIAL SE LLENO SOLO")
print("-" * 58)
paciente_p001 = clinica.buscar_paciente("72345678")
paciente_p001.historial.mostrar()
print("\nNadie escribio esas lineas a mano: las genero Cita.atender().")

# ---------- 8. Reporte ----------
print("\n8. REPORTE GENERAL")
print("-" * 58)
clinica.reporte_general()

EJECUCION DE PRUEBA DEL SISTEMA COMPLETO

1. REGISTRO DE DATOS
----------------------------------------------------------
Clinica San Gabriel (RUC 20512345678) 

Medico registrado: M001 - Dr(a). Carlos Rojas Vega
Medico registrado: M002 - Dr(a). Lucia Paredes Soto
Medico registrado: M003 - Dr(a). Javier Quispe Luna

Paciente registrado: P001 - Nahomi Anaya Yarleque
Paciente registrado: P002 - Ximena Guerra Vilchez
Paciente registrado: P003 - Sergio Mendoza Chavez
Paciente registrado: P004 - Dayvi Fernandez Leyva

2. REGISTROS INVALIDOS
----------------------------------------------------------
ERROR: el DNI debe tener 8 digitos numericos.
ERROR: ya existe un paciente con el DNI 72345678.
ERROR: la tarifa debe ser mayor que cero.

3. PROGRAMACION DE CITAS
----------------------------------------------------------
Cita C001 programada para el 26/10/2026 a las 09:00. Costo: S/ 48.0
Cita C002 programada para el 26/10/2026 a las 10:00. Costo: S/ 90
Cita C003 programada para el 27/10/2026 a 

## 4.3 De objetos a datos: conexión con pandas

El mismo camino de la práctica dirigida: **objetos → diccionarios → DataFrame → análisis**.

Vale la pena notar que `pandas` también está hecho con POO: `DataFrame` es una clase, `df` es un objeto, `df.shape` es un atributo y `df.head()` es un método.

In [11]:
# ------------------------------------------------------------
#  De objetos a datos: conexion con pandas
# ------------------------------------------------------------
# El mismo camino de la practica dirigida:
#    Objetos  ->  diccionarios  ->  DataFrame  ->  analisis

import pandas as pd

# Cada cita se convierte en un diccionario con a_diccionario()
df_citas = pd.DataFrame(clinica.tabla_citas())

print("TABLA DE CITAS")
print(df_citas)

print("\nTipo de df_citas:", type(df_citas))
print("Atributo .shape :", df_citas.shape)

print("\nCantidad de citas por estado:")
print(df_citas["estado"].value_counts())

print("\nIngreso por especialidad (solo citas atendidas):")
atendidas = df_citas[df_citas["estado"] == "ATENDIDA"]
print(atendidas.groupby("especialidad")["costo"].sum())

print("\nCosto promedio de una consulta atendida: S/", round(atendidas["costo"].mean(), 2))

TABLA DE CITAS
  codigo       fecha               paciente              medico  \
0   C001  26/10/2026  Nahomi Anaya Yarleque   Carlos Rojas Vega   
1   C002  26/10/2026  Ximena Guerra Vilchez  Lucia Paredes Soto   
2   C003  27/10/2026  Sergio Mendoza Chavez  Javier Quispe Luna   
3   C004  27/10/2026  Dayvi Fernandez Leyva   Carlos Rojas Vega   
4   C005  28/10/2026  Nahomi Anaya Yarleque  Javier Quispe Luna   
5   C006  27/10/2026  Ximena Guerra Vilchez   Carlos Rojas Vega   

       especialidad     estado  costo  
0  Medicina General   ATENDIDA   48.0  
1         Pediatria   ATENDIDA   90.0  
2       Cardiologia   ATENDIDA  120.0  
3  Medicina General  CANCELADA   60.0  
4       Cardiologia   ATENDIDA  120.0  
5  Medicina General  PENDIENTE   60.0  

Tipo de df_citas: <class 'pandas.DataFrame'>
Atributo .shape : (6, 7)

Cantidad de citas por estado:
estado
ATENDIDA     4
CANCELADA    1
PENDIENTE    1
Name: count, dtype: int64

Ingreso por especialidad (solo citas atendidas):
espec

## 4.4 Menú interactivo (requisito opcional)

Permite usar el sistema eligiendo opciones por número, sin escribir código.

**Para probarlo:** ejecute la celda, quite el `#` de la última línea y vuelva a ejecutarla. Salga con la opción `0`.

In [12]:
# ------------------------------------------------------------
#  Menu interactivo en consola (requisito opcional)
#  Integrante: Fernandez Leyva, Dayvi
# ------------------------------------------------------------
# Como todos los metodos validan sus datos e imprimen el error
# en lugar de detener el programa, el menu nunca se cae por un
# dato mal escrito: muestra el mensaje y vuelve a preguntar.


def mostrar_menu():
    print("")
    print("=" * 42)
    print("    CLINICA SAN GABRIEL - MENU PRINCIPAL")
    print("=" * 42)
    print("  1. Registrar paciente")
    print("  2. Registrar medico")
    print("  3. Listar pacientes")
    print("  4. Listar medicos")
    print("  5. Agendar cita")
    print("  6. Atender cita")
    print("  7. Cancelar cita")
    print("  8. Listar citas")
    print("  9. Ver historial de un paciente")
    print(" 10. Reporte general")
    print("  0. Salir")
    print("=" * 42)


def menu_principal(clinica):
    while True:
        mostrar_menu()
        opcion = input("Elija una opcion: ")

        if opcion == "1":
            nombre = input("Nombre              : ")
            apellido = input("Apellidos           : ")
            dni = input("DNI (8 digitos)     : ")
            telefono = input("Telefono (9 digitos): ")
            edad = int(input("Edad                : "))
            seguro = (input("Tiene seguro (s/n)  : ").lower() == "s")
            clinica.registrar_paciente(nombre, apellido, dni, telefono, edad, seguro)

        elif opcion == "2":
            nombre = input("Nombre              : ")
            apellido = input("Apellidos           : ")
            dni = input("DNI (8 digitos)     : ")
            telefono = input("Telefono (9 digitos): ")
            especialidad = input("Especialidad        : ")
            tarifa = float(input("Tarifa (S/)         : "))
            clinica.registrar_medico(nombre, apellido, dni, telefono, especialidad, tarifa)

        elif opcion == "3":
            print("")
            for paciente in clinica.pacientes:
                print(" ", paciente.mostrar_informacion())

        elif opcion == "4":
            print("")
            for medico in clinica.medicos:
                print(" ", medico.mostrar_informacion())

        elif opcion == "5":
            dni = input("DNI del paciente    : ")
            codigo = input("Codigo del medico   : ")
            fecha = input("Fecha (dd/mm/aaaa)  : ")
            hora = input("Hora (HH:MM)        : ")
            motivo = input("Motivo              : ")
            clinica.agendar_cita(dni, codigo, fecha, hora, motivo)

        elif opcion == "6":
            codigo = input("Codigo de la cita   : ")
            diagnostico = input("Diagnostico         : ")
            tratamiento = input("Tratamiento         : ")
            if clinica.atender_cita(codigo, diagnostico, tratamiento):
                print("")
                print(clinica.buscar_cita(codigo).comprobante())

        elif opcion == "7":
            codigo = input("Codigo de la cita   : ")
            clinica.cancelar_cita(codigo)

        elif opcion == "8":
            print("")
            for cita in clinica.listar_citas():
                print(" ", cita)

        elif opcion == "9":
            dni = input("DNI del paciente    : ")
            paciente = clinica.buscar_paciente(dni)
            if paciente is None:
                print("No existe un paciente con ese DNI.")
            else:
                print("")
                print(paciente.mostrar_informacion())
                paciente.historial.mostrar()

        elif opcion == "10":
            clinica.reporte_general()

        elif opcion == "0":
            print("\nGracias por usar el sistema. Hasta pronto.")
            break

        else:
            print("\nOpcion invalida. Elija un numero del menu.")


# Para probarlo: quite el # de la linea de abajo y ejecute la
# celda otra vez. Salga con la opcion 0.

# menu_principal(clinica)

---

## Conclusiones

1. **La herencia evitó repetir código.** Los cuatro datos personales están escritos una sola vez en `Persona`; `Paciente` y `Medico` los reciben con `super().__init__(...)`.
2. **Poner las reglas dentro de los métodos protegió al sistema.** `esta_disponible()` vive en `Medico` y `atender()` en `Cita`. Da igual desde dónde se llamen: la regla siempre se cumple. Por eso el menú no se cae con un dato mal escrito.
3. **El polimorfismo eliminó los `if` de tipo.** Un solo `for` recorre pacientes y médicos y cada objeto responde según su clase.
4. **La composición mantuvo el historial al día.** Como cada paciente contiene su historial, `atender()` lo escribe automáticamente.
5. **Los atributos de clase sirvieron para lo compartido:** los contadores generan los códigos correlativos y `descuento_seguro` guarda una política de la empresa en una sola línea.
6. **La POO se conecta con la Ciencia de Datos.** `a_diccionario()` convierte los objetos en registros y esos registros en un `DataFrame`.
